# 重新处理
## 密度修正
## 风速区间 风向区间 风频 发电量提升  加权发电量提升 

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
import os
import yaml
from constans import TURB_ATTRIBUTES_NEW, TURB_ATTRIBUTES_OLD, TURB_NUM
import xarray as xr

In [8]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)
all_entries = os.listdir(config['data']['splited_data_old'])

## 密度修正，全部修正到1.225标准密度下

In [4]:
ds = xr.open_dataset('./download_rou/dulan_rou.grib', engine='cfgrib') # 读取ERA5 文件
# print(ds)
point_data = ds.sel(longitude=96.15, latitude=36.3)
P_100m = point_data['sp'] - 1200 # 地表气压折算到100m
T_100m = point_data['t2m'] - 0.6 # 2m温度折算到100m
rho_hourly = P_100m / (287.05 * T_100m)
# 小时数据差值到分钟级
rho_minutely = rho_hourly.resample(time='1min').interpolate('linear')
# 转pandas
df_weather = rho_minutely.to_dataframe(name='actual_rou').reset_index()

In [5]:
def rou_coorrection(file_path, df_weather, standard_density=1.225, is_old_data=True):
    '''
    密度修正，并将天气数据合并到原始数据中，返回修正后的DataFrame
    根据新老数据切换数据标签
    '''
    df = pd.read_csv(file_path, parse_dates=['timestamp']) # 后面要合并，不设索引
    df_merged = pd.merge(df, df_weather, left_on='timestamp', right_on='time', how='inner') #按时间戳合并
    if 'time' in df_merged.columns and 'timestamp' in df_merged.columns:
        df_merged.drop(columns=['time','number', 'step', 'surface', 'latitude', 'longitude'], inplace=True)
    for i in range(1, TURB_NUM + 1):
        if is_old_data:
            df_rou_p = f'{TURB_ATTRIBUTES_OLD[2]}{i}'
        else:
            df_rou_p = f'{i}{TURB_ATTRIBUTES_NEW[1]}'
        df_merged[df_rou_p] = df_merged[df_rou_p] * (standard_density / df_merged['actual_rou'])
    return df_merged

In [11]:
save_path = config['data']['rou_corr_data_old']
standard_density = 1.225
for file_name in all_entries:
    print(f'正在处理文件: {file_name}')
    file_path = os.path.join(config['data']['splited_data_old'], file_name)
    df_merged = rou_coorrection(file_path, df_weather, standard_density, is_old_data=True)
    df_merged.to_csv(os.path.join(save_path, file_name), index=False,  encoding='utf-8-sig') 

正在处理文件: 2023-01.csv
正在处理文件: 2023-02.csv
正在处理文件: 2023-03.csv
正在处理文件: 2023-04.csv
正在处理文件: 2023-05.csv
正在处理文件: 2023-06.csv
正在处理文件: 2023-07.csv
正在处理文件: 2023-08.csv
正在处理文件: 2023-09.csv
正在处理文件: 2023-10.csv
正在处理文件: 2023-11.csv
正在处理文件: 2023-12.csv
正在处理文件: 2024-01.csv
正在处理文件: 2024-02.csv
正在处理文件: 2024-03.csv
正在处理文件: 2024-04.csv
正在处理文件: 2024-05.csv
正在处理文件: 2024-06.csv
正在处理文件: 2024-07.csv
正在处理文件: 2024-08.csv
正在处理文件: 2024-09.csv
正在处理文件: 2024-10.csv
正在处理文件: 2024-11.csv
正在处理文件: 2024-12.csv


## 所有数据进行风速区间 风向区间分类 并计算风频
按照时间顺序读入所有数据，将每条数据按照风速区间 风向区间归类 风速（3.5 0.5 14）风向(0 5 360)，不属于的归类到特殊区间

In [ ]:
ws_bins = np.arange(0, 20, 0.5)
wd_bins = np.arange(0, 365, 5)

def classify_ws_wd_optimized(file_path, save_dir, is_old_data=True):
    
    df = pd.read_csv(file_path, parse_dates=['timestamp'])
    df.set_index('timestamp', inplace=True)
    for i in range(1, TURB_NUM + 1):
        print(f'正在处理turb_{i}的数据...')
        if is_old_data:
            df_rol = [f'{attr}{i}' for attr in TURB_ATTRIBUTES_OLD]
            df_turb = df[df_rol].copy()
            df_turb_38 = df_turb[(df_turb.iloc[:, 3] == 5) & (df_turb.iloc[:, 4] == 0)].copy() # pandas不能使用and
            if df_turb_38.empty:
                continue
            # labels=ws_bins[:-1] 表示用区间的左端点作为标签（如 [3.5, 4.0) 的标签为 3.5）
            df_turb_38['ws_label'] = pd.cut(df_turb_38.iloc[:, 0], bins=ws_bins, labels=ws_bins[:-1], right=False)
            df_turb_38['wd_label'] = pd.cut(df_turb_38.iloc[:, 1], bins=wd_bins, labels=wd_bins[:-1], right=False)
        else:
            df_rol = [f'{i}{attr}' for attr in TURB_ATTRIBUTES_NEW]
            df_turb = df[df_rol].copy()
            df_turb_38 = df_turb[df_turb.iloc[:, 3] == 38].copy()
            if df_turb_38.empty:
                continue
            # labels=ws_bins[:-1] 表示用区间的左端点作为标签（如 [3.5, 4.0) 的标签为 3.5）
            df_turb_38['ws_label'] = pd.cut(df_turb_38.iloc[:, 4] , bins=ws_bins, labels=ws_bins[:-1], right=False)
            df_turb_38['wd_label'] = pd.cut(df_turb_38.iloc[:, 0] + df_turb_38.iloc[:, 3], bins=wd_bins, labels=wd_bins[:-1], right=False)
        # 先分组，后写入，较快
        buffer = {}
        for (ws_val, wd_val), group_df in df_turb_38.groupby(['ws_label', 'wd_label'], observed=True):
            if group_df.empty:
                continue
            csv_name = f'{ws_val}_{wd_val}_turb{i}.csv'
            path = os.path.join(save_dir, csv_name)
            # group_to_save = group_df.drop(columns=['ws_label', 'wd_label'])
            # header_flag = not os.path.exists(path) # 判断表头是否需要写入
            # group_to_save.to_csv(path, mode='a', header=header_flag, encoding='utf-8-sig')
            if csv_name not in buffer:  # 先保存，后写入，比较省时间
                buffer[csv_name] = []
            buffer[csv_name].append(group_df.drop(columns=['ws_label', 'wd_label']))
        for csv_name, df_list in buffer.items():
            path = os.path.join(save_dir, csv_name)
            final_group = pd.concat(df_list)
            header_flag = not os.path.exists(path)
            final_group.to_csv(path, mode='a', header=header_flag, encoding='utf-8-sig', index=True)


In [ ]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)
all_entries = os.listdir(config['data']['rou_corr_data_old'])
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 10, 31)
for file_name in all_entries:
    if not file_name.endswith('.csv'):
        continue
    try:
        date_str = file_name.split('.')[0]
        file_date = datetime.strptime(date_str, '%Y-%m')
        if start_date <= file_date <= end_date:
            print(f'符合条件，正在处理文件: {file_name}')
            file_path = os.path.join(config['data']['rou_corr_data_new'], file_name)
            save_dir = config['data']['classify_data_new']
            classify_ws_wd_optimized(file_path, save_dir, is_old_data=True)
        else:
            pass
    except ValueError:
        print(f'文件名 {file_name} 不符合日期格式，跳过处理。')

# 25年新数据处理

In [6]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)
all_entries = os.listdir(config['data']['splited_data_new'])

## 25年新数据密度修正

In [8]:
save_path = config['data']['rou_corr_data_new']
standard_density = 1.225
for file_name in all_entries:
    print(f'正在处理文件: {file_name}')
    file_path = os.path.join(config['data']['splited_data_new'], file_name)
    df_merged = rou_coorrection(file_path, df_weather, standard_density, is_old_data=False)
    df_merged.to_csv(os.path.join(save_path, file_name), index=False,  encoding='utf-8-sig') 

正在处理文件: 2024-11.csv
正在处理文件: 2024-12.csv
正在处理文件: 2025-01.csv
正在处理文件: 2025-02.csv
正在处理文件: 2025-03.csv
正在处理文件: 2025-04.csv
正在处理文件: 2025-05.csv
正在处理文件: 2025-06.csv
正在处理文件: 2025-07.csv
正在处理文件: 2025-08.csv
正在处理文件: 2025-09.csv
正在处理文件: 2025-10.csv
正在处理文件: 2025-11.csv
正在处理文件: 2025-12.csv
正在处理文件: 2026-01.csv


## 25年数据统计

In [ ]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)
all_entries = os.listdir(config['data']['rou_corr_data_new'])
start_date = datetime(2024, 11, 1) 
end_date = datetime(2025, 12, 31) 
for file_name in all_entries:
    if not file_name.endswith('.csv'):
        continue
    try:
        date_str = file_name.split('.')[0]
        file_date = datetime.strptime(date_str, '%Y-%m')
        if start_date <= file_date <= end_date:
            print(f'符合条件，正在处理文件: {file_name}')
            file_path = os.path.join(config['data']['rou_corr_data_new'], file_name)
            save_dir = config['data']['classify_data_new']
            classify_ws_wd_optimized(file_path, save_dir, is_old_data=False)
        else:
            pass
    except ValueError:
        print(f'文件名 {file_name} 不符合日期格式，跳过处理。')